In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import os

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    f1_score,
    precision_recall_curve,
    auc
)

components = [
    'LVEF < 50 or RWMA',
    'Mod to Large Pericardial Effusion',
    'Intracardiac mass',
    'Acute Aortic Syndrome',
    'ASD/VSD',
    '2+ or greater VHD',
    'Moderate or greater PH',
    'Aortic aneurysm',
    'RV function'
]

n_components = len(components)
print("Number of outcomes:", n_components)


In [ ]:
model_path = os.getcwd()+ "/models/"
path = model_path + 'ECG_Echo_CNN_MC/'
y_train = np.load(path + 'y_train.npy',allow_pickle=True)
y_test = np.load(path + 'y_test.npy',allow_pickle=True)
y_val = np.load(path + 'y_val.npy',allow_pickle=True)
y_pred_train = np.load(path + 'y_pred_train.npy',allow_pickle=True)
y_pred_test = np.load(path + 'y_pred_test.npy',allow_pickle=True)
y_pred_val = np.load(path + 'y_pred_val.npy',allow_pickle=True)

print("Y train:", y_train.shape)
print("Y val  :", y_val.shape)
print("Y test :", y_test.shape)

print("Y pred train:", y_pred_train.shape)
print("Y pred val  :", y_pred_val.shape)
print("Y pred test :", y_pred_test.shape)

In [ ]:
def prevalence_table(y, components):
    rows = []
    for i, component in enumerate(components):

        valid = ~np.isnan(y[:, i])
        yy = y[valid, i]

        n = len(yy)
        positive = np.sum(yy == 1)
        negative = np.sum(yy == 0)

        rows.append({
            'Component': component,
            'N': n,
            'Positive': positive,
            'Negative': negative,
            'Prevalence (%)':
                positive / n * 100 if n > 0 else np.nan
        })
    return pd.DataFrame(rows)

print("\nTRAIN")
print(prevalence_table(y_train, components).round(2))

print("\nVALIDATION")
print(prevalence_table(y_val, components).round(2))

print("\nTEST")
print(prevalence_table(y_test, components).round(2))

In [ ]:
def sklearn_metrics_return(
    y_true,
    y_pred,
    threshold=0.5
):

    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()

    valid = ~np.isnan(y_true)

    y_true = y_true[valid]
    y_pred = y_pred[valid]

    y_bin = np.copy(y_pred)
    y_bin[y_bin >= threshold] = 1
    y_bin[y_bin < threshold] = 0

    positive = np.sum(y_true == 1)
    negative = np.sum(y_true == 0)

    TP = np.sum(
        (y_true == 1) &
        (y_bin == 1)
    )

    TN = np.sum(
        (y_true == 0) &
        (y_bin == 0)
    )

    FP = np.sum(
        (y_true == 0) &
        (y_bin == 1)
    )

    FN = np.sum(
        (y_true == 1) &
        (y_bin == 0)
    )

    sensitivity = (
        TP / (TP + FN)
        if (TP + FN) > 0
        else np.nan
    )

    specificity = (
        TN / (TN + FP)
        if (TN + FP) > 0
        else np.nan
    )

    if positive > 0 and negative > 0:
        auc_score = roc_auc_score(
            y_true,
            y_pred
        )
    else:
        auc_score = np.nan

    if positive > 0:
        precision, recall, thresholds = (
            precision_recall_curve(
                y_true,
                y_pred
            )
        )

        auprc = auc(
            recall,
            precision
        )
    else:
        auprc = np.nan

    accuracy = accuracy_score(
        y_true,
        y_bin
    )

    macro_f1 = f1_score(
        y_true,
        y_bin,
        average='macro',
        zero_division=0
    )

    return {
        'N': len(y_true),
        'Positive': positive,
        'Negative': negative,
        'Prevalence (%)':
            positive / len(y_true) * 100
            if len(y_true) > 0
            else np.nan,
        'Accuracy': accuracy ,
        'AUC': auc_score ,
        'AUPRC': auprc ,
        'Macro F1': macro_f1 ,
        'Sensitivity': sensitivity ,
        'Specificity': specificity ,
        'TP': TP,
        'TN': TN,
        'FP': FP,
        'FN': FN
    }


# Calculate metrics for EACH component
def calc_metrics(y_true, y_pred, thr):
    results =[];
    
    for i, component in enumerate(components):
    
        metrics = sklearn_metrics_return(
            y_true[:, i],       # TRUE value for component i
            y_pred[:, i],       # PREDICTION for component i
            threshold=thr
        )
    
        metrics['Component'] = component
        results.append(metrics)
    
    component_results = pd.DataFrame(results)
    component_results = component_results[
        [
            'Component',
            'N',
            'Positive',
            'Negative',
            'Prevalence (%)',
            'Accuracy',
            'AUC',
            'AUPRC',
            'Macro F1',
            'Sensitivity',
            'Specificity',
            'TP',
            'TN',
            'FP',
            'FN'
        ]
    ]
    
    pd.set_option(
        'display.max_columns',
        None
    )
    
    print(
        component_results.round(2)
    )
     
    component_results.round(2).to_csv(
        'component_model_performance.csv',
        index=False
    )

In [ ]:
calc_metrics(y_test, y_pred_test, 0.5)

In [ ]:
calc_metrics(y_test, y_pred_test, 0.119)